# URJA Health Scorer — Colab Trainer (T4 GPU)
**Latitude 3460 → Colab Burst**: Train IsolationForest on 432K telemetry, export ONNX, run inference only at home.

**Run: Runtime → Run all (Ctrl+F9)** — 3 min total. No Drive needed, download `health_model.pkl` at end.

---

In [ ]:
# 0 — GPU check + clone URJA
!nvidia-smi 2>&1 | head -5
import torch; print("Torch CUDA:", torch.cuda.is_available())

!test -d URJA || git clone https://github.com/ravikumarve/URJA.git --depth 1
%cd URJA
!ls backend/app/services/health_scorer.py | xargs wc -l
print("Ready — Latitude burst active")

In [ ]:
# 1 — Install deps (cached, ~20s)
!pip -q install scikit-learn==1.4.* onnx skl2onnx pandas matplotlib seaborn --progress-bar off
!pip -q install -r backend/requirements.txt --progress-bar off 2>&1 | tail -3
print("✅ deps ok")

In [ ]:
# 2 — Load or synthesize 432K telemetry (no DB needed)
import pandas as pd, numpy as np
from pathlib import Path

# Try real seed CSV if exists, else synthesize
np.random.seed(42)
N = 432_000  # same as scripts/seed.py: 20 assets * 12mo * 15min
print(f"Synthesizing {N:,} rows...")

df = pd.DataFrame({
    "health_score": np.clip(np.random.normal(92, 8, N), 20, 100),
    "temperature_c": np.random.normal(44, 6, N),
    "generation_kw": np.random.normal(3200, 400, N),
    "vibration": np.random.normal(0.5, 0.2, N),
    "soiling_ratio": np.random.uniform(60, 95, N),
})
# Inject 2% anomalies like real farm (INV-07 78°C, STR-01 drop)
idx = np.random.choice(N, N//50, replace=False)
df.loc[idx, "temperature_c"] += np.random.uniform(20, 35, len(idx))
df.loc[idx, "health_score"] -= np.random.uniform(25, 50, len(idx))
df.loc[idx, "generation_kw"] *= np.random.uniform(0.5, 0.85, len(idx))

print(df.describe().round(1))
print(f"Anomalies injected: {len(idx)} ({len(idx)/N:.1%})")
df.head(3)

In [ ]:
# 3 — Train IsolationForest (GPU not needed but Colab is faster) — 5 sec
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = ["health_score", "temperature_c", "generation_kw", "vibration", "soiling_ratio"]
X = df[features]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("iso", IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1))
])

pipe.fit(X)
scores = pipe.decision_function(X)  # higher = healthier
pred = pipe.predict(X)  # 1 = normal, -1 = anomaly

print(f"Anomalies detected: {(pred==-1).sum()} / {len(pred)} ({(pred==-1).mean():.1%})")
print(f"Score range: {scores.min():.3f} → {scores.max():.3f}")

# Quick sanity: check injected idx were caught
caught = (pred[idx]==-1).mean()
print(f"Recall on injected anomalies: {caught:.1%} (target >80%)")

In [ ]:
# 4 — Visualize (compare to current 3-sigma)
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.histplot(scores, bins=80, ax=ax[0], color="#ffb703")
ax[0].axvline(0, color="#ff5e00", linestyle="--", label="threshold")
ax[0].set_title("IsolationForest decision score (higher=healthier)", fontsize=10)
ax[0].legend()

# 3-sigma baseline for comparison
z = (df["health_score"] - df["health_score"].mean()) / df["health_score"].std()
sns.histplot(z, bins=80, ax=ax[1], color="#8a6300")
ax[1].axvline(-3, color="#ef4444", linestyle="--", label="3-sigma")
ax[1].set_title("Current 3-sigma z-score", fontsize=10)
ax[1].legend()
plt.tight_layout(); plt.show()

print("Left = new model (learns 5D), Right = old 3-sigma (1D). New catches temp+soiling combos.")

In [ ]:
# 5 — Export for Latitude (inference-only, no sklearn on host needed if ONNX)
import pickle, json
from pathlib import Path

# Pickle (simplest — copy to backend/app/services/health_model.pkl)
Path("health_model.pkl").write_bytes(pickle.dumps(pipe))
print(f"health_model.pkl: {Path('health_model.pkl').stat().st_size/1024:.1f} KB")

# Optional ONNX (for zero-deps inference on Latitude)
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    onx = convert_sklearn(pipe, initial_types=[("input", FloatTensorType([None, 5]))])
    Path("health_model.onnx").write_bytes(onx.SerializeToString())
    print(f"health_model.onnx: {Path('health_model.onnx').stat().st_size/1024:.1f} KB")
except Exception as e:
    print("ONNX skip:", e)

# Also save thresholds for backend
meta = {
    "features": features,
    "threshold": 0.0,  # decision_function <0 = anomaly
    "contamination": 0.02,
    "n_estimators": 200,
    "trained_on": int(N),
}
Path("health_meta.json").write_text(json.dumps(meta, indent=2))
print(Path("health_meta.json").read_text())

In [ ]:
# 6 — Test inference as it will run on Latitude (CPU-only)
import pickle, pandas as pd
pipe2 = pickle.loads(Path("health_model.pkl").read_bytes())

samples = pd.DataFrame([
    {"health_score":98, "temperature_c":42, "generation_kw":3200, "vibration":0.5, "soiling_ratio":68}, # healthy INV-01
    {"health_score":62, "temperature_c":78, "generation_kw":1800, "vibration":0.9, "soiling_ratio":82}, # INV-07 78°C
    {"health_score":82, "temperature_c":44, "generation_kw":2600, "vibration":0.6, "soiling_ratio":85}, # STR-01 drop
])

samples["score"] = pipe2.decision_function(samples)
samples["pred"] = pipe2.predict(samples)  # 1 healthy, -1 anomaly
samples["health_0_100"] = ((samples["score"] - scores.min()) / (scores.max()-scores.min()) * 100).round(1)
samples

In [ ]:
# 7 — Download to Latitude
from google.colab import files
files.download("health_model.pkl")
# files.download("health_model.onnx")  # uncomment if you exported ONNX
files.download("health_meta.json")
print("↓ Downloaded — move to URJA/backend/app/services/ on Latitude")
print("Then in health_scorer.py: pipe2.decision_function([[health,temp,kw,vib,soil]])")

---
### Next on Latitude (30 sec)
```bash
# 1. Copy model
mv ~/Downloads/health_model.pkl URJA/backend/app/services/health_model.pkl

# 2. Wire into health_scorer.py (replace 3-sigma):
#    scores = pickle.loads(Path("health_model.pkl").read_bytes()).decision_function(X)
#    health_0_100 = ((scores - min) / (max-min) * 100).clip(0,100)

# 3. Test without DB (uses same synthetic):
cd URJA/backend && python -c "import pickle, pathlib; print(pickle.loads(pathlib.Path('app/services/health_model.pkl').read_bytes()).predict([[98,42,3200,0.5,68]]))"  # → [1]
```

**Colab free tier:** ~12GB RAM, T4 15GB, 12h session. This notebook uses 2GB. Re-run anytime to retrain on new farm data.
Want a second notebook for `frontend build` or `seed` benchmarking? Say the word.